# Maximum Mean Discrepancy in the Walsh–Hadamard Space

## Theoretical Framework

The exact Maximum Mean Discrepancy (MMD) between two probability distributions $ p $ and $ q $ over bitstrings can be expressed in the Walsh–Hadamard (WH) basis as:

$$
MMD^2(p,q) = \sum_{\lambda \in \{0,1\}^n} w_\sigma(|\lambda|)\, \Delta_\lambda^2(p,q)
$$

where:
- $ \lambda \in \{0,1\}^n $ labels Walsh–Hadamard frequencies (equivalently, diagonal Pauli-$Z$ operators),
- $ |\lambda| $ is the Hamming weight (correlation order),
- $ \Delta_\lambda(p,q) := \hat{p}_\lambda - \hat{q}_\lambda $,
- $ w_\sigma(|\lambda|) $ is a binomial weight induced by the Gaussian kernel.

This representation reveals that MMD acts as a **spectral filter** over correlation orders:
- Low Hamming weight → low-order (local) correlations,
- High Hamming weight → high-order (global) correlations.



## Monte Carlo Estimation in WH Space

The exact computation involves $2^n$ terms and is intractable for large $n$. In practice, the loss is estimated via Monte Carlo sampling:

$$
\widehat{MMD}^2 = \frac{1}{n_{\text{ops}}} \sum_{k=1}^{n_{\text{ops}}} \Delta_{\lambda_k}^2(p,q)
$$
CONTROLLARE DOVE FINISCE OMEGA
where:
- $ \lambda_k \sim w_\sigma $ are sampled via a binomial process,
- sampling is implemented through `jax.random.binomial(..., p_MMD)`.

This estimator is **unbiased**, and its variance is controlled by $ n_{\text{ops}} $.



## Role of the Parameters
The learning dynamics arise from the interaction of:

- **Soft filter:** $ \sigma $ (what is measured),
- **Hard constraint:** `max_weight` (what can be generated),
- **Estimator variance:** `n_ops`,
- **Data noise:** `n_samples`.
- **Data entropy** `entropy`

A key phenomenon is **spectral mismatch**:
- If the loss emphasizes frequencies the model cannot represent, optimization fails or saturates.


### Kernel Parameter $ \sigma $ (Soft Spectral Filter)

The Gaussian kernel bandwidth $ \sigma $ determines the sampling probability `p_MMD`, and thus the spectral region emphasized:

- **Large $ \sigma $** (small `p_MMD`):  
  Emphasizes low Hamming weights → local correlations (low-pass filter).

- **Small $ \sigma $** (`p_MMD \approx 0.5`):  
  Peaks around $ |\lambda| \approx n/2 $ → global correlations (high-pass / band-pass filter).



### Number of Operators (`n_ops`) — Monte Carlo Variance

- Controls the number of sampled WH frequencies per optimization step.
- Does **not** truncate frequencies.

Regimes:
- Small `n_ops`: high-variance (noisy gradients),
- Large `n_ops`: accurate approximation of exact MMD.

Interpretation: analogous to mini-batch size in stochastic optimization.


### Maximum Weight (`max_weight`) — Architectural Expressivity

Defines the maximal order of interactions in the IQP circuit:
- Example: `max_weight = 2` → only terms like $ e^{i\theta Z_i Z_j} $.

This imposes a **hard cutoff**:
- The model cannot generate independent high-weight correlations,
- Limits the accessible WH spectrum.


### Number of Samples (`n_samples`) — Empirical Bias (Shot Noise)

Finite sampling introduces noise in estimating:

$$
\Delta_\lambda = \langle Z_\lambda \rangle_p - \langle Z_\lambda \rangle_q
$$

Effect:
- Adds approximately white noise across all frequencies,
- Introduces an asymptotic bias:

$$
\mathcal{O}\left(\frac{1}{s}\right)
$$

Implication:
- High-frequency (small signal) components are drowned by sampling noise.



### Target Entropy — Spectral Structure

The entropy of the target distribution determines its WH spectrum:

- **Low entropy (structured / peaked):**  
  Strong correlations → spectral weight at higher $ |\lambda| $

- **High entropy (near-uniform):**  
  Weak correlations → spectrum concentrated at low weights




## Experiments
### Spectral Sensitivity vs Entropy
**Goal:** Study interaction between entropy and $ \sigma $.
- Generate targets with varying entropy,
- Sweep $ \sigma $.

**Hypothesis:**
- Low-entropy targets → signal persists even at small $ \sigma $,
- High-entropy targets → gradients vanish as $ \sigma $ decreases.


<img src="./1_exp.png" alt="Plot for 1st Experiment" width="700"/>

### Empirical Bias vs Number of Samples
**Goal:** Verify $ \mathcal{O}(1/s) $ scaling.
- Compute MMD between:
  - a dataset,
  - a resampled version of itself.
- Fix:
  - large `n_ops`,
  - fixed $ \sigma $,
- Vary `n_samples` logarithmically.
**Expected result:**
- Log-log plot of $ \widehat{MMD}^2 $ vs $ s $,
- Slope $ \approx -1 $.

### Expressivity vs Spectral Filter
**Goal:** Test mismatch between `max_weight` and $ \sigma $.

- Fix a **low-entropy target** (rich correlations),
- Use IQP model with `max_weight = 2`.

Train with:
1. **Large $ \sigma $** (low-pass):
   - Model should converge.
2. **Small $ \sigma $** (high-pass):
   - Model cannot match required correlations.

**Hypothesis:**  
Training fails or saturates at suboptimal loss for small $ \sigma $.

### Stochastic Efficiency (`n_ops`)

**Goal:** Determine minimal viable Monte Carlo resolution.

- Fix stable regime (`σ`, `max_weight`, `n_samples`),
- Train with decreasing `n_ops`: $1000 \to 1$.

**Hypothesis:**
- Gradients remain unbiased,
- Below a critical `n_ops`, variance prevents convergence.


## Practical Guidelines

- Use **large $ \sigma $** when model expressivity is limited (low-pass filter)
- `n_ops` is connected with the gradiets magnitude! Increase until gradients stabilize
- Ensure `n_samples` is sufficient to resolve high-frequency signals,
- Match $ \sigma $ to target entropy,